In [ ]:
# parameters
# BINDER_FAST: set N=6, n_eps=8, k_max=3 for fast cloud execution
N = 8                # Hilbert space truncation
omega0_GHz = 5.0     # bare 0->1 frequency (GHz)
K_GHz = 0.2          # Kerr nonlinearity (GHz)
kappa_MHz = 1.0      # peak single-photon loss rate (MHz)
Gamma_MHz = 100.0    # bath FWHM (MHz)
nbar = 0.02          # thermal occupation
gamma_phi_MHz = 0.05 # pure dephasing rate (MHz)
Delta_MHz = 5.0      # drive detuning (ω_d - ω₀) / 2π (MHz)
n_eps = 20           # number of points in epsilon sweep


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
import scipy.integrate
%matplotlib widget

from bosonic_gates.driven_kerr import (
    DrivenKerrConfig,
    make_operators,
    J, J_vectorized,
    compute_floquet_modes,
    compute_fourier_components,
    assemble_rates_with_dephasing,
    run_floquet_markov,
    run_full_floquet_markov,
    floquet_steady_state,
    effective_decay_rate_from_R,
    run_lindblad,
    run_redfield,
    effective_loss_rate,
    extract_excited_pop,
)
from bosonic_gates.driven_kerr import validation

TWO_PI = 2 * np.pi


## Module 3c: Floquet–Markov Theory for Driven Dissipative Systems

**Learning objectives:**
- Understand the four-step Floquet–Markov (FM) pipeline: modes → Fourier components → rates → dynamics
- Visualize quasi-energies and Floquet mode structure
- Explain how the Floquet sideband shift suppresses the decay rate at strong drive
- Reproduce the three-method rate comparison showing a 20× discrepancy at $\varepsilon = K$
- Decompose the Method C rate into loss, thermal, and drive-induced dephasing channels
- Run the 7-check validation suite

---

**Sections:**
[1 Floquet Theory for Periodic Systems](#sec1) · 
[2 Fourier Components and Rate Assembly](#sec2) · 
[3 Full Rate vs. Drive Amplitude](#sec3) · 
[4 Error Budget](#sec4) · 
[5 Validation Suite](#sec5) · 
[6 Exercises](#sec6)


<a id="sec1"></a>
## 1  Floquet Theory for Periodic Systems

### 1.1  Floquet theorem

For a Hamiltonian with exact time periodicity $\hat{H}(t) = \hat{H}(t+T_d)$
(with $T_d = 2\pi/\omega_d$), Floquet's theorem guarantees that the
time-evolution operator has the decomposition:

$$\hat{U}(t, 0) = \sum_m |\phi_m(t)\rangle e^{-i\varepsilon_m t}\langle\phi_m(0)|$$

where the **Floquet modes** $|\phi_m(t)\rangle = e^{-i\varepsilon_m t}|u_m(t)\rangle$
are quasiperiodic: $|u_m(t)\rangle = |u_m(t+T_d)\rangle$.
The **quasi-energies** $\varepsilon_m$ are defined modulo $\omega_d$.

### 1.2  Computing Floquet modes (Step 1)

We build the one-period propagator $\hat{U}(T_d, 0)$ column-by-column:
propagate each basis state $|n\rangle$ forward by one drive period and collect
the results into a matrix.  Diagonalizing this matrix gives eigenvalues
$e^{-i\varepsilon_m T_d}$ from which the quasi-energies $\varepsilon_m$ are extracted.

The dressed Floquet modes are **superpositions of Fock states** — as $\epsilon$
increases the modes mix more and more Fock content.

*Lab note: the Floquet picture is the natural language for periodically driven
quantum systems. In the microwave context, quasi-energies are the "dressed" resonance
frequencies of the cavity–drive system. They are the quantities measured by
two-tone spectroscopy after a strong pump tone is applied.*


In [ ]:
# Build canonical configuration
cfg = DrivenKerrConfig(
    N        = N,
    omega0   = TWO_PI * omega0_GHz,
    K        = TWO_PI * K_GHz,
    omega_d  = TWO_PI * (omega0_GHz + Delta_MHz * 1e-3),  # slightly blue-detuned
    omega_f  = TWO_PI * omega0_GHz,                        # bath at omega0
    epsilon  = TWO_PI * 0.5 * K_GHz,                      # representative: 0.5K
    kappa    = TWO_PI * kappa_MHz * 1e-3,
    Gamma    = TWO_PI * Gamma_MHz * 1e-3,
    nbar     = nbar,
    gamma_phi= TWO_PI * gamma_phi_MHz * 1e-3,
    k_max    = 5,
    n_t      = 512,
)

a_op, adag_op, n_op = make_operators(cfg.N)

print(f"Configuration: ω₀/2π = {cfg.omega0/TWO_PI:.3f} GHz")
print(f"  K/2π = {cfg.K/TWO_PI*1e3:.1f} MHz,  ε = {cfg.epsilon/cfg.K:.2f} K")
print(f"  ω_d - ω₀ = {(cfg.omega_d-cfg.omega0)/TWO_PI*1e3:.1f} MHz (detuning Δ)")
print(f"  T_d = {cfg.T_d*1e9:.4f} ns")

print("\nStep 1: computing Floquet modes (this may take ~30 s for N=8, n_t=512)...")
modes_t, quasi_energies, tgrid = compute_floquet_modes(cfg)
print(f"Done. modes_t.shape = {modes_t.shape}  (N_modes × n_t × N_hilbert)")
print(f"quasi_energies.shape = {quasi_energies.shape}")


In [ ]:
# Display quasi-energies vs bare Fock energies
H0 = cfg.omega0 * n_op - (cfg.K / 2) * adag_op * adag_op * a_op * a_op
bare_energies = H0.eigenenergies()

print("Quasi-energies vs. bare Fock level energies:")
print(f"{'Mode':>5}  {'ε_m / 2π (GHz)':>18}  {'bare E_n / 2π (GHz)':>22}  {'Dominant |n⟩':>14}")
for m in range(cfg.N):
    # Identify which Fock state this mode overlaps most with at t=0
    overlaps = np.abs(modes_t[m, 0, :])**2
    dominant_n = int(np.argmax(overlaps))
    print(f"{m:>5}  {quasi_energies[m]/TWO_PI:>18.4f}  "
          f"{bare_energies[dominant_n]/TWO_PI:>22.4f}  "
          f"{dominant_n:>14}  (overlap {overlaps[dominant_n]:.3f})")


In [ ]:
# Visualize Floquet mode composition (Fock weights) at t=0
fig, axes = plt.subplots(1, min(4, cfg.N), figsize=(12, 3), sharey=False)
n_show = min(4, cfg.N)

for m in range(n_show):
    ax = axes[m]
    weights = np.abs(modes_t[m, 0, :])**2
    ax.bar(range(cfg.N), weights, color="steelblue", alpha=0.8)
    ax.set_xlabel("Fock state $|n\\rangle$")
    ax.set_title(rf"Floquet mode $m={m}$" + "\n"
                 + rf"$\varepsilon_m/2\pi = {quasi_energies[m]/TWO_PI:.3f}$ GHz")
    if m == 0:
        ax.set_ylabel(r"$|\langle n|\phi_m(0)\rangle|^2$")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    rf"Floquet mode Fock composition at $t=0$, $\varepsilon = {cfg.epsilon/cfg.K:.2f}\,K$",
    y=1.02
)
plt.tight_layout()
plt.show()

print("\nAt weak drive the Floquet modes look like pure Fock states.")
print("At ε = 0.5K the modes are already visibly mixed.")


<a id="sec2"></a>
## 2  Fourier Components and Rate Assembly

### 2.1  Floquet sideband matrix elements (Step 2)

The bath coupling in the Floquet basis is via the operator $\hat{a}$ (annihilation).
The time-dependent matrix element between Floquet modes $m$ and $n$ is:

$$S_{mn}(t) = \langle\phi_m(t)|\hat{a}|\phi_n(t)\rangle.$$

Because both $|\phi_m(t)\rangle$ and $|\phi_n(t)\rangle$ are periodic with period $T_d$,
$S_{mn}(t)$ is also periodic.  Its Fourier series:

$$S_{mn}(t) = \sum_k S_{mn}^{(k)} e^{-ik\omega_d t},
\quad S_{mn}^{(k)} = \frac{1}{T_d}\int_0^{T_d} e^{+ik\omega_d t} S_{mn}(t)\,dt.$$

Each Fourier component $S_{mn}^{(k)}$ is a **sideband coupling amplitude** at
sideband order $k$.

### 2.2  Rate matrix assembly (Step 3)

The Floquet–Markov transition rate from mode $n$ to mode $m$ is:

$$\Gamma_{mn} = \sum_k |S_{mn}^{(k)}|^2\,J(\Delta_{mn} - k\omega_d)$$

where $\Delta_{mn} = \varepsilon_m - \varepsilon_n$ is the quasi-energy difference.

**Physical mechanism for rate suppression:** as $\varepsilon$ increases,
the quasi-energy difference $\Delta_{m_0, m_1}$ between the modes with highest
overlap with $|0\rangle$ and $|1\rangle$ shifts.  The dominant $k = -1$ sideband
samples the bath at frequency $\Delta_{m_0, m_1} + \omega_d$, which shifts
by $\approx \varepsilon^2/\Gamma$ away from the bath center $\omega_f = \omega_0$.
When this shift exceeds $\Gamma/2$, the bath coupling is exponentially suppressed.


In [ ]:
# Step 2: compute Fourier components for the loss operator a
print("Step 2: computing Fourier components...")
a_matrix = a_op.full()
S_k = compute_fourier_components(modes_t, a_matrix, cfg, tgrid)

print(f"Fourier components S_mn^(k) for k in [{-cfg.k_max}, ..., {cfg.k_max}]:")
print(f"  dict keys: {sorted(S_k.keys())}")
print(f"  S_k[-1] shape: {S_k[-1].shape}  (N×N matrix)")

# Identify the Floquet modes most similar to |0> and |1> at t=0
mode_fock0 = int(np.argmax([abs(modes_t[m, 0, 0])**2 for m in range(cfg.N)]))
mode_fock1 = int(np.argmax([abs(modes_t[m, 0, 1])**2 for m in range(cfg.N)]))
print(f"\nMode most like |0⟩: m = {mode_fock0} "
      f"(overlap {abs(modes_t[mode_fock0,0,0])**2:.4f})")
print(f"Mode most like |1⟩: m = {mode_fock1} "
      f"(overlap {abs(modes_t[mode_fock1,0,1])**2:.4f})")

print(f"\nSideband amplitudes |S_{{m0,m1}}^(k)|² for loss (m0={mode_fock0}, m1={mode_fock1}):")
for k in range(-cfg.k_max, cfg.k_max + 1):
    amp_sq = abs(S_k[k][mode_fock0, mode_fock1])**2
    freq = (quasi_energies[mode_fock0] - quasi_energies[mode_fock1] - k * cfg.omega_d) / TWO_PI
    j_val = J(quasi_energies[mode_fock0] - quasi_energies[mode_fock1] - k * cfg.omega_d, cfg)
    print(f"  k={k:+2d}: |S|² = {amp_sq:.5f},  freq = {freq:.3f} GHz,  J = {j_val/TWO_PI*1e3:.4f} MHz")


In [ ]:
# Step 3: assemble rate matrix
print("Step 3: assembling rate matrix R...")
R = assemble_rates_with_dephasing(modes_t, quasi_energies, cfg, tgrid)
print(f"R shape: {R.shape}")
print(f"\nRate matrix R (off-diagonal elements, in MHz / 2π):")
R_MHz = R / TWO_PI * 1e3
for i in range(min(4, cfg.N)):
    row = [f"{R_MHz[i,j]:8.4f}" for j in range(min(4, cfg.N))]
    print("  " + "  ".join(row))

print(f"\nDominant transition (m1→m0):")
print(f"  R[{mode_fock0},{mode_fock1}] / 2π = {R[mode_fock0, mode_fock1]/TWO_PI*1e3:.4f} MHz")
print(f"  Compare to Method A rate: {J(cfg.omega0, cfg)/TWO_PI*1e3:.4f} MHz")

# Steady state
p_ss = floquet_steady_state(R)
print(f"\nSteady-state populations (Floquet basis):")
for m in range(min(4, cfg.N)):
    print(f"  mode {m}: p_ss = {p_ss[m]:.4f}")
print(f"  Sum = {p_ss.sum():.6f} (should be 1.0)")


The key number is `R[mode_fock0, mode_fock1]` — the rate from the mode resembling
$|1\rangle$ to the mode resembling $|0\rangle$.  At $\epsilon = 0.5K$ this is already
noticeably below the bare $\kappa / 2\pi = 1.0$ MHz.  At $\epsilon = K$ it will be
about 30× smaller.

**Why does the $k=-1$ sideband dominate?**
The coupling operator $\hat{a}$ lowers by one photon, so its dominant matrix element
in the Floquet basis connects quasi-energy levels separated by approximately
$\varepsilon_{m_0} - \varepsilon_{m_1} \approx -\omega_0$.  The $k = -1$ Fourier
component shifts the sampled frequency to
$\Delta_{m_0, m_1} - (-1)\omega_d = (\varepsilon_{m_0} - \varepsilon_{m_1}) + \omega_d \approx \omega_0$,
which sits near the bath center.  As the drive increases, this sideband frequency
walks off toward higher frequencies, suppressing $J$.


<a id="sec3"></a>
## 3  Full Rate vs. Drive Amplitude — Three-Method Comparison

We now reproduce the key Figure 5a result from the reference paper:
sweep $\epsilon$ from $0.001K$ to $1.5K$ and compare the effective decay rate
for all three methods.

**Expected result:**
- **Method A (Lindblad):** flat at $\approx \kappa(1+2\bar{n}) \approx 1.04$ MHz
- **Method B (Redfield):** nearly identical to A (narrow bath, $\Gamma \ll \omega_d$)
- **Method C (Floquet–Markov):** starts near 0.97 MHz at weak drive, then drops
  monotonically to $\approx 0.03$ MHz at $\epsilon = K$ — a **factor of ~35** suppression

The anti-crossing feature in Method C near $\epsilon \approx 40$–50 MHz is a physical
Floquet level near-degeneracy, not a numerical artifact.


In [ ]:
# Epsilon sweep
K_rad = cfg.K
eps_arr = np.geomspace(1e-3 * K_rad, 1.5 * K_rad, n_eps)

rates_A = np.full(n_eps, np.nan)
rates_C = np.full(n_eps, np.nan)
leak_C  = np.full(n_eps, np.nan)

# Method A is analytical (no simulation needed)
for i, eps in enumerate(eps_arr):
    cfg_i = cfg.replace(epsilon=eps)
    rates_A[i] = J(cfg_i.omega0, cfg_i) + J(-cfg_i.omega0, cfg_i)

print(f"Sweeping {n_eps} ε values from {eps_arr[0]/TWO_PI*1e3:.3f} to {eps_arr[-1]/TWO_PI*1e3:.1f} MHz")
print("Method C requires Floquet mode computation per point — may take several minutes.")

for i, eps in enumerate(eps_arr):
    cfg_i = cfg.replace(epsilon=eps)
    try:
        modes_t_i, qe_i, tgrid_i = compute_floquet_modes(cfg_i)
        N_i = cfg_i.N

        # Identify Floquet modes with highest overlap with |0> and |1>
        m0_i = int(np.argmax([abs(modes_t_i[m, 0, 0])**2 for m in range(N_i)]))
        m1_i = int(np.argmax([abs(modes_t_i[m, 0, 1])**2 for m in range(N_i)]))

        if m0_i == m1_i:
            print(f"  eps={eps/TWO_PI*1e3:.1f} MHz: mode collision (NaN)")
            continue

        R_i = assemble_rates_with_dephasing(modes_t_i, qe_i, cfg_i, tgrid_i)
        rates_C[i] = max(0.0, R_i[m0_i, m1_i])

        # Steady-state leakage
        p0_fm = np.zeros(N_i); p0_fm[m1_i] = 1.0
        t_ss = 20.0 / max(np.abs(np.diag(R_i)).max(), 1e-30)
        sol = scipy.integrate.solve_ivp(
            lambda t, p: R_i @ p, (0.0, t_ss), p0_fm,
            method="RK45", rtol=1e-8, atol=1e-10,
        )
        p_ss_i = np.abs(sol.y[:, -1]); p_ss_i /= p_ss_i.sum()
        leak_C[i] = max(0.0, 1.0 - p_ss_i[m0_i] - p_ss_i[m1_i])

    except Exception as e:
        print(f"  eps={eps/TWO_PI*1e3:.1f} MHz: {e}")

    if (i + 1) % 5 == 0 or i == n_eps - 1:
        print(f"  {i+1}/{n_eps} done")

print("Sweep complete.")


In [ ]:
# Plot: rate vs epsilon (log-log, both panels)
eps_MHz = eps_arr / TWO_PI * 1e3
K_MHz = K_rad / TWO_PI * 1e3
kappa_MHz_val = cfg.kappa / TWO_PI * 1e3

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel (a): decay rate
ax = axes[0]
valid_C = np.isfinite(rates_C) & (rates_C > 0)

ax.loglog(eps_MHz, rates_A / TWO_PI * 1e3, "C0-o", ms=5, lw=1.5,
          label="A (Lindblad)")
ax.loglog(eps_MHz[valid_C], rates_C[valid_C] / TWO_PI * 1e3,
          "C2-^", ms=5, lw=1.5, label="C (Floquet–Markov)")
ax.axhline(kappa_MHz_val, color="gray", ls="--", lw=0.9,
           label=rf"$\kappa/2\pi = {kappa_MHz_val:.2f}$ MHz")
ax.axvline(K_MHz, color="k", ls=":", lw=0.8, alpha=0.6, label=r"$\varepsilon = K$")

ax.set_xlabel(r"Drive amplitude $\varepsilon/2\pi$ (MHz)")
ax.set_ylabel(r"Decay rate $\Gamma/2\pi$ (MHz)")
ax.set_title("(a) Effective decay rate vs. drive amplitude")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")

# Annotate ratio at ε = K
idx_K = np.argmin(np.abs(eps_arr - K_rad))
if valid_C[idx_K]:
    ratio = rates_A[idx_K] / rates_C[idx_K]
    ax.annotate(
        rf"$\Gamma_A / \Gamma_C \approx {ratio:.0f}\times$ at $\varepsilon = K$",
        xy=(K_MHz, rates_C[idx_K] / TWO_PI * 1e3),
        xytext=(K_MHz * 0.3, rates_C[idx_K] / TWO_PI * 1e3 * 4),
        fontsize=9, color="C2",
        arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
    )

# Panel (b): leakage
ax = axes[1]
valid_L = np.isfinite(leak_C) & (leak_C > 1e-12)
if valid_L.any():
    ax.loglog(eps_MHz[valid_L], leak_C[valid_L],
              "C2-^", ms=5, lw=1.5, label="C (Floquet–Markov)")
ax.axvline(K_MHz, color="k", ls=":", lw=0.8, alpha=0.6)
ax.text(0.95, 0.07, "A, B: $L_1 \\equiv 0$",
        transform=ax.transAxes, fontsize=9, color="gray",
        ha="right", va="bottom")
ax.set_xlabel(r"Drive amplitude $\varepsilon/2\pi$ (MHz)")
ax.set_ylabel(r"Steady-state leakage $L_1$")
ax.set_title("(b) Leakage (Method C only; A and B give $L_1 = 0$)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print(f"Key numbers at ε = K ({K_MHz:.0f} MHz):")
if valid_C[idx_K]:
    print(f"  Γ_A / 2π = {rates_A[idx_K]/TWO_PI*1e3:.4f} MHz (flat)")
    print(f"  Γ_C / 2π = {rates_C[idx_K]/TWO_PI*1e3:.4f} MHz (suppressed)")
    print(f"  Ratio Γ_A / Γ_C = {rates_A[idx_K]/rates_C[idx_K]:.1f}×")
if valid_L[idx_K]:
    print(f"  Leakage L₁ = {leak_C[idx_K]:.3%}")


**Mechanism explained:**
As $\varepsilon$ increases, the dressed quasi-energy difference
$\Delta_{m_0, m_1} = \varepsilon_{m_0} - \varepsilon_{m_1}$ shifts.
The dominant sideband ($k = -1$) samples the bath at frequency
$\Delta_{m_0, m_1} + \omega_d$, which shifts away from the bath center $\omega_f = \omega_0$
by approximately $\varepsilon^2 / \Gamma$.  When this shift exceeds $\sim \Gamma/2$,
the Lorentzian bath spectral density $J$ evaluated there falls sharply,
suppressing the decay rate.

Methods A and B are **structurally blind** to this shift because they do not use
the dressed basis — they always sample $J$ at the bare frequency $\omega_0$.


<a id="sec4"></a>
## 4  Error Budget at $\varepsilon = 0.5K$

We decompose the Method C rate at $\varepsilon = 0.5K$ into three channels:

1. **Loss** ($\kappa \neq 0$, $\bar{n} = 0$, $\gamma_\phi = 0$): photon emission into bath
2. **Thermal** ($\kappa \neq 0$, $\bar{n} \neq 0$ minus loss-only): bath photon absorption
3. **Dephasing** ($\kappa = 0$, $\gamma_\phi \neq 0$): pure dephasing

**Expected result from reference paper (Figure 6a):**

| Channel | Method A | Method C |
|---------|----------|----------|
| Loss | 1.02 MHz | **0.030 MHz** |
| Thermal | 0.020 MHz | 0.00075 MHz |
| Dephasing | **0** (exact) | **0.013 MHz** (drive-induced!) |

The dephasing contribution in Method C is drive-induced — $\gamma_\phi$ conserves
photon number but the Floquet modes mix Fock states, so the number-conserving
dephasing channel couples to population transfer.
**Drive-induced dephasing accounts for ~43% of the total Method C rate.**


In [ ]:
# Error budget at epsilon = 0.5K
eps_budget = 0.5 * cfg.K
cfg_budget = cfg.replace(epsilon=eps_budget)

print(f"Running error budget at ε = 0.5K = {eps_budget/TWO_PI*1e3:.1f} MHz...")

# Compute Floquet modes once (shared)
modes_t_b, qe_b, tgrid_b = compute_floquet_modes(cfg_budget)
m0_b = int(np.argmax([abs(modes_t_b[m, 0, 0])**2 for m in range(cfg.N)]))
m1_b = int(np.argmax([abs(modes_t_b[m, 0, 1])**2 for m in range(cfg.N)]))

def rate_C_channel(kappa_val, nbar_val, gphi_val):
    cfg_c = cfg_budget.replace(kappa=kappa_val, nbar=nbar_val, gamma_phi=gphi_val)
    R_c = assemble_rates_with_dephasing(modes_t_b, qe_b, cfg_c, tgrid_b)
    return max(0.0, float(R_c[m0_b, m1_b]))

rate_C_total   = rate_C_channel(cfg_budget.kappa, cfg_budget.nbar, cfg_budget.gamma_phi)
rate_C_loss    = rate_C_channel(cfg_budget.kappa, 0.0, 0.0)            # kappa only, no nbar
rate_C_no_thm  = rate_C_channel(cfg_budget.kappa, 0.0, cfg_budget.gamma_phi)  # no thermal
rate_C_thermal = rate_C_total - rate_C_no_thm
rate_C_deph    = rate_C_channel(0.0, cfg_budget.nbar, cfg_budget.gamma_phi)  # dephasing + thermal

# Method A (analytical)
rate_A_loss    = J(cfg_budget.omega0, cfg_budget)
rate_A_thermal = J(-cfg_budget.omega0, cfg_budget)
rate_A_deph    = 0.0
rate_A_total   = rate_A_loss + rate_A_thermal

print(f"\nError budget at ε/2π = {eps_budget/TWO_PI*1e3:.1f} MHz (= 0.5K):")
print(f"{'Channel':<14} {'Method A (MHz)':>18} {'Method C (MHz)':>18}")
print("-" * 52)
for ch, rA, rC in [
    ("Loss",     rate_A_loss,    rate_C_loss),
    ("Thermal",  rate_A_thermal, rate_C_thermal),
    ("Dephasing",rate_A_deph,    rate_C_deph),
    ("Total",    rate_A_total,   rate_C_total),
]:
    print(f"{ch:<14} {rA/TWO_PI*1e3:>18.5f} {rC/TWO_PI*1e3:>18.5f}")

deph_frac = rate_C_deph / rate_C_total if rate_C_total > 0 else 0
print(f"\nDrive-induced dephasing fraction: {deph_frac:.1%} of total Method C rate")


In [ ]:
# Bar chart
channels = ["loss", "thermal", "deph"]
vals_A_MHz = [rate_A_loss/TWO_PI*1e3, rate_A_thermal/TWO_PI*1e3, rate_A_deph]
vals_C_MHz = [rate_C_loss/TWO_PI*1e3, rate_C_thermal/TWO_PI*1e3, rate_C_deph/TWO_PI*1e3]

x = np.arange(len(channels))
w = 0.35
floor = 1e-5

fig, ax = plt.subplots(figsize=(6, 4))
bars_A = ax.bar(x - w/2, [max(v, floor) for v in vals_A_MHz], w,
                color="C0", alpha=0.85, label="Method A (Lindblad)")
bars_C = ax.bar(x + w/2, [max(v, floor) for v in vals_C_MHz], w,
                color="C2", alpha=0.85, label="Method C (Floquet–Markov)")

# Annotate Method A dephasing = 0 (exact)
ax.text(x[2] - w/2, floor * 2, "0 (exact)", ha="center", va="bottom",
        fontsize=8, color="C0")

ax.set_yscale("log")
ax.set_ylim(bottom=floor * 0.3)
ax.set_xticks(x)
ax.set_xticklabels(["Loss", "Thermal", "Dephasing"])
ax.set_ylabel(r"Rate contribution / $2\pi$ (MHz)")
ax.set_title(
    rf"Error budget at $\varepsilon = 0.5K = {eps_budget/TWO_PI*1e3:.0f}$ MHz\n"
    f"Drive-induced dephasing = {deph_frac:.0%} of Method C total"
)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


**Key findings from the error budget:**

1. **Loss channel:** Method A assigns $\sim 1.02$ MHz; Method C assigns $\sim 0.030$ MHz.
   This is a **factor of ~35** difference. Method A is wrong in the strong-drive regime.

2. **Dephasing channel:** Method A assigns exactly **zero** to the population decay from
   dephasing — correct for the undriven Fock basis (the $\hat{n}$ jump operator commutes with
   $\hat{n}$ and cannot change photon number).  Method C shows a **non-zero contribution**
   from dephasing.  This is a purely drive-induced effect: $\gamma_\phi\hat{n}$ in the
   Floquet basis has off-diagonal elements that couple the modes resembling $|0\rangle$ and
   $|1\rangle$, because these modes are Fock-state superpositions.  This is the
   **drive-induced dephasing** mechanism.

3. **Method C total rate at $\varepsilon = 0.5K$** is about 20× below Method A,
   meaning Lindblad overestimates the decoherence rate by this factor.


<a id="sec5"></a>
## 5  Validation Suite

Before trusting any simulation result, we run the 7-check validation suite.
All checks must pass (or be marked SOFT WARN for the Redfield positivity check,
which is expected to fail at strong drive).

| Check | Description |
|-------|-------------|
| 7.1 Fock convergence | Rate stable as N increases (N=6→8→12) |
| 7.2 Weak-drive A vs C | Methods A and C agree at ε → 0 (flat bath) |
| 7.3 k_max convergence | Rates stable as sideband truncation increases |
| 7.4 fmmesolve cross-check | Hand-built FM rates match QuTiP (if available) |
| 7.5 White-bath control | A and C agree for flat bath at large detuning |
| 7.6 Thermal limit | Steady state matches Boltzmann distribution |
| 7.7 Redfield positivity | Redfield does not produce large negative populations |


In [ ]:
# Run full validation suite (will print detailed output)
# Use a small cfg for speed (N=4 for the fast tutorial run)
cfg_val = cfg.replace(N=4, n_t=128, k_max=3)
print("Running validation suite (N=4 for speed; use N=8 for full validation)...")
print("This may take 2–5 minutes.\n")

results_val = validation.run_all(cfg_val)

print("\nValidation summary:")
all_ok = True
for name, passed in results_val.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"  {status}  {name}")
print(f"\nOverall: {'ALL PASSED' if all_ok else 'SOME FAILURES (investigate before trusting results)'}")


<a id="sec6"></a>
## 6  Exercises

### Exercise 1: White bath convergence

Repeat the rate sweep from Section 3 with a very wide bath:
$\Gamma / 2\pi = 500$ MHz (instead of 100 MHz).

Do Methods A and C converge for the wide bath?
If not, explain why a wide bath alone is not sufficient to make
Lindblad exact (hint: there is also a **matrix-element redistribution mechanism** —
the Floquet modes mix Fock states, so the dressed $|1\rangle\to|0\rangle$ coupling
amplitude is smaller than the bare one even for a flat bath).

**Expected:** partial convergence for small $\varepsilon$, but a residual 10–30%
discrepancy persists at $\varepsilon \sim K$ even for the white bath, because
Fock-state mixing in the Floquet modes redistributes the coupling amplitude
across sidebands regardless of bath shape.


In [ ]:
# YOUR CODE HERE
# Hint:
# cfg_wide = cfg.replace(Gamma=TWO_PI * 500e-3, omega_f=cfg.omega0)  # 500 MHz FWHM
# Then repeat the epsilon sweep for Method C rates_C_wide
# and compare to rates_A (Method A is unchanged -- it still samples at omega0)


### Exercise 2: Fock convergence check

Run the rate sweep at $N = 6, 8, 12$ and compare the Method C rate at $\varepsilon = K$.
How large is the relative change when $N$ is doubled?

**Expected:** rates should converge to within $< 0.1\%$ between $N = 8$ and $N = 12$,
confirming that the Hilbert space truncation is not affecting the result.

**Caution:** $N = 12$ with `n_t = 512` is slow; use `n_t = 128` or `k_max = 3`
for the convergence check.


In [ ]:
# YOUR CODE HERE
# Hint: loop over N_list = [6, 8, 12]
# For each N, build cfg_N = cfg.replace(N=N_val, n_t=128, k_max=3)
# then compute the FM rate at eps = K and compare


In [ ]:
# Solution (hidden in PDF output)
N_list = [6, 8]  # skip 12 in tutorial to save time
rates_N = {}
eps_test = cfg.K  # test at ε = K

for N_val in N_list:
    cfg_N = cfg.replace(N=N_val, epsilon=eps_test, n_t=128, k_max=3)
    try:
        modes_t_n, qe_n, tgrid_n = compute_floquet_modes(cfg_N)
        m0_n = int(np.argmax([abs(modes_t_n[m, 0, 0])**2 for m in range(N_val)]))
        m1_n = int(np.argmax([abs(modes_t_n[m, 0, 1])**2 for m in range(N_val)]))
        if m0_n == m1_n:
            rates_N[N_val] = np.nan
            continue
        R_n = assemble_rates_with_dephasing(modes_t_n, qe_n, cfg_N, tgrid_n)
        rates_N[N_val] = max(0.0, float(R_n[m0_n, m1_n]))
    except Exception as e:
        rates_N[N_val] = np.nan
        print(f"N={N_val}: {e}")

print(f"Fock convergence check at ε = K = {eps_test/TWO_PI*1e3:.1f} MHz:")
for N_val, rate in rates_N.items():
    print(f"  N={N_val}: Γ_C/2π = {rate/TWO_PI*1e3:.5f} MHz")

vals = [v for v in rates_N.values() if np.isfinite(v)]
if len(vals) >= 2:
    rel = abs(vals[-1] - vals[-2]) / abs(vals[-2])
    print(f"Relative change: {rel:.3%}")
